- 2025-5-26
- Applying the BOOMER Python API

Currently, using the latest version of Python package

- Update at 2025-7-1:
Currently, need update BOOMER as Version 0.12.0

In [2]:
#!/usr/bin/env python
#-*- coding:utf-8 -*-

import numpy as np
import os, sys
from datetime import date
from mlrl.boosting import Boomer

sys.path.append("..")
from data import Data
from options import Options
from boomerer import Boomerer

In [3]:
# 250926: prepare the name of the output folder with the date info
boomer_output = 'temp_boomers_' + date.today().isoformat()[2:].replace('-', '')

# 2025/6/9: trying Boomer using boomerer.py
# The default split ration is 20% for testing accuracy
bench_multi = '../bench_multi'
dataset_list = [os.path.join(bench_multi, n, n + '.csv') for n in sorted(os.listdir(bench_multi))]
nb_rules_candidate = list(range(10, 101, 10))
result_json = []

# 250926: add the parameter of maximum conditions used (equivalent max_depth for xgboost)
max_conds = [4, 5, 6]

wdigits = [None, 1, 2]

def train_specific_boomer(dataset, max_rule, out_path, max_cond, wdigit=None):
    # preparing the command line for options
    # '-t' indicates training, '-n' indicates the n_estimator
    # 2511: add the option of limiting the weight digits
    if not wdigit is None:
        command_line = 'xxx -o ' + out_path + ' -t -n ' + str(max_rule) + ' -d ' + str(max_cond) + ' --wdigit ' + str(wdigit) + ' --drule ' + dataset
    else:
        command_line = 'xxx -o ' + out_path + ' -t -n ' + str(max_rule) + ' -d ' + str(max_cond) + ' --drule ' + dataset
    options = Options(command_line.split())

    if options.files:
        boomer = None
        if options.train:
            data = Data(filename=options.files[0], mapfile=options.mapfile,
                    separator=options.separator,
                    use_categorical=options.use_categorical)

            boomer = Boomerer(options, from_data=data)

            train_acc, test_acc, mod_fpath, num_rules = boomer.train()
            return train_acc, test_acc, mod_fpath, boomer.nb_features, boomer.num_class, boomer.num_instance, num_rules


In [3]:

for dataset in dataset_list:
    for nb_rule in nb_rules_candidate:
        for max_cond in max_conds:
            for wd in wdigits:
                # print("dataset {}, nb_rule {}, max_con {}".format(dataset, nb_rule, max_cond))
                train_acc, test_acc, mod_fpath, nb_feat, nb_c, nb_i, nb_rules = train_specific_boomer(dataset, nb_rule, boomer_output, max_cond, wd)
                dataset_n = dataset.split('/')[-1].split('.')[0]
                res_dict = {'dataset': dataset_n, 'n_estimator': nb_rule, 'max_depth': max_cond,\
                            'train_acc': train_acc, 'test_acc': test_acc, 'num_feat': nb_feat,\
                            'num_class': nb_c, 'num_instance': nb_i, 'num_identical_path': nb_rules,\
                            'wdigit': wd}
                result_json.append(res_dict)


In [5]:
# 260209: just train Boomer for Iris dataset, and visualize the tree structure
iris_csv = '../bench_multi/Iris.csv'
n, d = 2, 2

train_acc, test_acc, mod_fpath, _,_,_,_ = train_specific_boomer(iris_csv, n, 'iris_boomer', d)

In [4]:
import json
json_file_name = 'result' + boomer_output[4:] + '.json'
if os.path.isfile(json_file_name):
    os.remove(json_file_name)
with open(json_file_name, 'w') as f:
    json.dump(result_json, f, indent=2)